In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
# Image Deduplication Pipeline for 1.5M Images

This notebook implements a two-stage filtering system:
1. **Stage 1**: Perceptual hashing (dHash) to remove exact duplicates
2. **Stage 2**: CLIP embeddings + FAISS to remove semantically similar images

**Optimized for Google Colab A100 GPU**

## Setup Instructions
1. Upload this notebook to Google Colab
2. Select Runtime → Change runtime type → A100 GPU
3. Mount your Google Drive or upload images to Colab storage
4. Update `RAW_IMAGES_DIR` in the configuration cell

## 1. Mount Google Drive (Optional)

If your images are stored in Google Drive, mount it here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify mount
!ls /content/drive/MyDrive

## 2. Install Required Packages

In [ ]:
!pip install webdataset open_clip_torch faiss-gpu imagehash pillow tqdm -q

# Verify GPU
!nvidia-smi

## 3. Import Libraries and Configuration

In [ ]:
import os
import json
import tarfile
import hashlib
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool, cpu_count
from typing import List, Dict, Tuple

import numpy as np
import torch
import faiss
import imagehash
from PIL import Image
from tqdm.auto import tqdm
import webdataset as wds
from torch.utils.data import DataLoader
import open_clip

# Configuration - UPDATE THESE PATHS FOR YOUR SETUP
# Option 1: Using Google Drive
RAW_IMAGES_DIR = f"{DRIVE_ROOT}/RawImages"
TAR_OUTPUT_DIR = f"{DRIVE_ROOT}/tar_shards"

# Option 2: Using Colab local storage (faster but limited space ~100GB)
# RAW_IMAGES_DIR = "/content/RawImages"
# TAR_OUTPUT_DIR = "/content/tar_shards"

HASH_RESULTS_FILE = "/content/hash_results.json"
FINAL_RESULTS_FILE = "/content/unique_images.json"

# Parameters
IMAGES_PER_SHARD = 10000  # Number of images per tar file
HASH_SIZE = 8  # Size for dHash (8x8 = 64 bits)
SIMILARITY_THRESHOLD = 0.98  # Cosine similarity threshold for CLIP embeddings
BATCH_SIZE = 512  # Batch size for CLIP inference (can increase to 1024+ on A100)
NUM_WORKERS = 4  # DataLoader workers

print(f"✓ Libraries imported")
print(f"✓ CPU cores available: {cpu_count()}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 4. Stage 0: Create TAR Shards from Raw Images

Pack images into TAR archives to avoid small-file I/O bottleneck.

**Note**: This step can take a while for 1.5M images. The TAR files will be saved to Google Drive or local Colab storage depending on your configuration.

In [ ]:
def collect_all_images(base_dir: str) -> List[str]:
    """Collect all image paths from the directory structure."""
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    all_images = []
    
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if Path(file).suffix.lower() in image_extensions:
                all_images.append(os.path.join(root, file))
    
    return all_images

def create_tar_shards(image_paths: List[str], output_dir: str, images_per_shard: int = 10000):
    """Create TAR archives containing image shards."""
    os.makedirs(output_dir, exist_ok=True)
    
    total_images = len(image_paths)
    num_shards = (total_images + images_per_shard - 1) // images_per_shard
    
    shard_info = []
    
    for shard_idx in tqdm(range(num_shards), desc="Creating TAR shards"):
        shard_name = f"images_shard_{shard_idx:05d}.tar"
        shard_path = os.path.join(output_dir, shard_name)
        
        start_idx = shard_idx * images_per_shard
        end_idx = min(start_idx + images_per_shard, total_images)
        shard_images = image_paths[start_idx:end_idx]
        
        with tarfile.open(shard_path, 'w') as tar:
            for img_idx, img_path in enumerate(shard_images):
                # Use original path as key for tracking
                arcname = f"{shard_idx:05d}_{img_idx:06d}{Path(img_path).suffix}"
                tar.add(img_path, arcname=arcname)
                shard_info.append({
                    'shard': shard_name,
                    'arcname': arcname,
                    'original_path': img_path
                })
    
    # Save mapping for later reference
    with open(os.path.join(output_dir, 'shard_mapping.json'), 'w') as f:
        json.dump(shard_info, f, indent=2)
    
    print(f"✓ Created {num_shards} TAR shards with {total_images} images")
    return shard_info

# Collect and create shards
print("Collecting images from:", RAW_IMAGES_DIR)
all_image_paths = collect_all_images(RAW_IMAGES_DIR)
print(f"Found {len(all_image_paths)} images")

if len(all_image_paths) > 0:
    shard_mapping = create_tar_shards(all_image_paths, TAR_OUTPUT_DIR, IMAGES_PER_SHARD)
else:
    print("⚠ No images found. Check the RAW_IMAGES_DIR path.")

## 5. Stage 1: Perceptual Hashing (dHash) - CPU Parallel

Compute dHash for all images using multiprocessing to filter exact/near duplicates.

In [ ]:
def compute_hash_from_tar(args):
    """Compute dHash for a single TAR shard."""
    tar_path, shard_name = args
    results = []
    
    try:
        with tarfile.open(tar_path, 'r') as tar:
            members = tar.getmembers()
            for member in members:
                if member.isfile():
                    try:
                        f = tar.extractfile(member)
                        img = Image.open(f).convert('RGB')
                        img_hash = str(imagehash.dhash(img, hash_size=HASH_SIZE))
                        results.append({
                            'arcname': member.name,
                            'shard': shard_name,
                            'hash': img_hash
                        })
                    except Exception as e:
                        print(f"Error processing {member.name}: {e}")
    except Exception as e:
        print(f"Error opening {tar_path}: {e}")
    
    return results

def deduplicate_by_hash(hash_results: List[Dict]) -> List[Dict]:
    """Keep only the first occurrence of each hash."""
    seen_hashes = set()
    unique_images = []
    duplicates = []
    
    for item in hash_results:
        if item['hash'] not in seen_hashes:
            seen_hashes.add(item['hash'])
            unique_images.append(item)
        else:
            duplicates.append(item)
    
    return unique_images, duplicates

# Process all TAR shards in parallel
tar_files = sorted([f for f in os.listdir(TAR_OUTPUT_DIR) if f.endswith('.tar')])
tar_paths = [(os.path.join(TAR_OUTPUT_DIR, f), f) for f in tar_files]

print(f"Processing {len(tar_files)} TAR shards with {cpu_count()} CPU cores...")

all_hashes = []
with Pool(processes=cpu_count()) as pool:
    results = list(tqdm(
        pool.imap(compute_hash_from_tar, tar_paths),
        total=len(tar_paths),
        desc="Computing hashes"
    ))
    for result in results:
        all_hashes.extend(result)

print(f"✓ Computed {len(all_hashes)} hashes")

# Deduplicate
unique_after_hash, hash_duplicates = deduplicate_by_hash(all_hashes)
print(f"✓ Unique images after hashing: {len(unique_after_hash)}")
print(f"✓ Duplicates removed: {len(hash_duplicates)}")

# Save hash results
with open(HASH_RESULTS_FILE, 'w') as f:
    json.dump(unique_after_hash, f, indent=2)

print(f"✓ Hash results saved to {HASH_RESULTS_FILE}")

## 6. Stage 2: CLIP Embedding Extraction (A100 GPU)

Extract embeddings for remaining unique images using CLIP with optimized settings.

In [ ]:
def create_filtered_tar_shards(unique_images: List[Dict], output_dir: str) -> List[str]:
    """Create new TAR shards containing only unique images after hash filtering."""
    filtered_dir = os.path.join(output_dir, "filtered_shards")
    os.makedirs(filtered_dir, exist_ok=True)
    
    # Group by original shard
    shard_groups = defaultdict(list)
    for img in unique_images:
        shard_groups[img['shard']].append(img)
    
    filtered_shards = []
    for shard_name, images in tqdm(shard_groups.items(), desc="Creating filtered shards"):
        original_tar = os.path.join(output_dir, shard_name)
        filtered_tar = os.path.join(filtered_dir, shard_name)
        
        arcnames_to_keep = {img['arcname'] for img in images}
        
        with tarfile.open(original_tar, 'r') as src_tar:
            with tarfile.open(filtered_tar, 'w') as dst_tar:
                for member in src_tar.getmembers():
                    if member.name in arcnames_to_keep:
                        f = src_tar.extractfile(member)
                        dst_tar.addfile(member, f)
        
        filtered_shards.append(filtered_tar)
    
    return filtered_shards

# Create filtered TAR shards for CLIP processing
print("Creating filtered TAR shards...")
filtered_tar_paths = create_filtered_tar_shards(unique_after_hash, TAR_OUTPUT_DIR)
print(f"✓ Created {len(filtered_tar_paths)} filtered shards")

In [ ]:
# Load CLIP model
print("Loading CLIP model...")
model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', 
    pretrained='laion2b_s34b_b79k'
)
model = model.cuda().eval()

# Optional: Compile for extra speed (PyTorch 2.0+)
try:
    model = torch.compile(model)
    print("✓ Model compiled with torch.compile()")
except Exception as e:
    print(f"⚠ torch.compile() not available: {e}")

print("✓ CLIP model loaded on GPU")

In [ ]:
# Create WebDataset pipeline
filtered_shard_dir = os.path.join(TAR_OUTPUT_DIR, "filtered_shards")
tar_pattern = os.path.join(filtered_shard_dir, "images_shard_*.tar")

dataset = (
    wds.WebDataset(tar_pattern)
    .decode("pil")
    .to_tuple("jpg;jpeg;png;bmp;webp", "__key__")
    .map_tuple(preprocess, lambda x: x)
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# Extract embeddings
all_embeddings = []
all_keys = []

print(f"Extracting CLIP embeddings (batch_size={BATCH_SIZE})...")

with torch.no_grad(), torch.cuda.amp.autocast():
    for images, keys in tqdm(loader, desc="Processing batches"):
        images = images.cuda()
        features = model.encode_image(images)
        features = features / features.norm(dim=-1, keepdim=True)  # Normalize for cosine similarity
        
        all_embeddings.append(features.cpu().numpy())
        all_keys.extend(keys)

# Concatenate all embeddings
embeddings = np.concatenate(all_embeddings, axis=0)
print(f"✓ Extracted {embeddings.shape[0]} embeddings (shape: {embeddings.shape})")

# Save embeddings for later use
np.save("embeddings.npy", embeddings)
with open("embedding_keys.json", "w") as f:
    json.dump(all_keys, f, indent=2)

print("✓ Embeddings saved to embeddings.npy")

## 7. Stage 3: FAISS Similarity Search (GPU)

Use FAISS to find and remove semantically similar images based on embedding similarity.

In [ ]:
# Build FAISS index on GPU
print("Building FAISS index on GPU...")

embedding_dim = embeddings.shape[1]
res = faiss.StandardGpuResources()

# Use IndexFlatIP for exact cosine similarity (Inner Product on normalized vectors)
index = faiss.IndexFlatIP(embedding_dim)
gpu_index = faiss.index_cpu_to_gpu(res, 0, index)

# Add embeddings to index
gpu_index.add(embeddings.astype(np.float32))

print(f"✓ FAISS index built with {gpu_index.ntotal} vectors")

# Search for similar images
print(f"Searching for similar pairs (threshold={SIMILARITY_THRESHOLD})...")
k = 10  # Find top 10 most similar images for each image

distances, indices = gpu_index.search(embeddings.astype(np.float32), k)

print(f"✓ Search complete")
print(f"Distance matrix shape: {distances.shape}")  # (N, k)

In [ ]:
# Deduplicate based on similarity threshold
def deduplicate_by_similarity(distances, indices, threshold=0.98):
    """
    Remove duplicates based on cosine similarity threshold.
    For each image, if it has a neighbor with similarity > threshold,
    mark the higher-indexed one as a duplicate.
    """
    n = distances.shape[0]
    to_remove = set()
    
    for i in tqdm(range(n), desc="Finding duplicates"):
        if i in to_remove:
            continue
            
        # Skip first result (self-match with distance ~1.0)
        for j in range(1, distances.shape[1]):
            neighbor_idx = indices[i, j]
            similarity = distances[i, j]
            
            # If similarity exceeds threshold and neighbor not already marked
            if similarity >= threshold and neighbor_idx not in to_remove:
                # Remove the higher index to maintain deterministic order
                to_remove.add(max(i, neighbor_idx))
    
    # Create list of unique indices
    unique_indices = [i for i in range(n) if i not in to_remove]
    return unique_indices, to_remove

unique_indices, duplicate_indices = deduplicate_by_similarity(
    distances, 
    indices, 
    SIMILARITY_THRESHOLD
)

print(f"✓ Images after similarity filtering: {len(unique_indices)}")
print(f"✓ Duplicates removed: {len(duplicate_indices)}")

# Map back to original keys
final_unique_keys = [all_keys[i] for i in unique_indices]

print(f"✓ Final unique images: {len(final_unique_keys)}")

## 8. Save Final Results

Map keys back to original file paths and save the list of unique images.

In [ ]:
# Load shard mapping to get original paths
with open(os.path.join(TAR_OUTPUT_DIR, 'shard_mapping.json'), 'r') as f:
    shard_mapping = json.load(f)

# Create a lookup dictionary: arcname -> original_path
arcname_to_path = {item['arcname']: item['original_path'] for item in shard_mapping}

# Map final unique keys to original paths
final_results = []
for key in final_unique_keys:
    # key format from webdataset: "shard_name/arcname" or just "arcname"
    # We need to match it with arcname_to_path
    arcname = key.split('/')[-1] if '/' in key else key
    
    if arcname in arcname_to_path:
        final_results.append({
            'arcname': arcname,
            'original_path': arcname_to_path[arcname]
        })
    else:
        # Fallback: just store the key
        final_results.append({
            'arcname': key,
            'original_path': 'unknown'
        })

# Save final results
with open(FINAL_RESULTS_FILE, 'w') as f:
    json.dump(final_results, f, indent=2)

print(f"✓ Final results saved to {FINAL_RESULTS_FILE}")
print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total images processed: {len(all_image_paths)}")
print(f"After hash filtering: {len(unique_after_hash)}")
print(f"After embedding filtering: {len(final_unique_keys)}")
print(f"Total duplicates removed: {len(all_image_paths) - len(final_unique_keys)}")
print(f"{'='*60}")

## 9. Optional: Export Unique Images to New Directory

Copy the final unique images to a new directory for easy access.

**Warning**: This can take significant time and storage for large datasets. Consider exporting only if needed.

In [ ]:
import shutil

def export_unique_images(results, output_dir):
    """Copy unique images to a new directory."""
    os.makedirs(output_dir, exist_ok=True)
    
    for i, item in enumerate(tqdm(results, desc="Exporting images")):
        src_path = item['original_path']
        if os.path.exists(src_path):
            # Create unique filename to avoid collisions
            ext = Path(src_path).suffix
            dst_filename = f"unique_{i:07d}{ext}"
            dst_path = os.path.join(output_dir, dst_filename)
            
            shutil.copy2(src_path, dst_path)
    
    print(f"✓ Exported {len(results)} unique images to {output_dir}")

# Uncomment to export (UPDATE PATH FOR COLAB)
# EXPORT_DIR = f"{DRIVE_ROOT}/UniqueImages"
# export_unique_images(final_results, EXPORT_DIR)

## 10. Download Results to Local Machine

Download the JSON files containing the results.

In [ ]:
from google.colab import files

# Download result files
files.download(FINAL_RESULTS_FILE)
files.download(HASH_RESULTS_FILE)

# Optional: Download embeddings if needed
# files.download("embeddings.npy")
# files.download("embedding_keys.json")